# Session 25: Linear, Ridge, Lasso, and Elastic Net

## Objective

Train four baseline regression models using the full-information
feature scenario and compare their test performance using MAE, RMSE,
and R-squared.

Models:

1. Linear Regression
2. Ridge Regression
3. Lasso Regression
4. Elastic Net Regression

The main comparison metric is RMSE. Lower RMSE indicates better
test-set predictive performance.

This notebook is designed for local execution in VS Code. It does not
use Google Colab.


## 1. Imports and Project Paths


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

TABLES_DIRECTORY = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIRECTORY = PROJECT_ROOT / "reports" / "figures"

TABLES_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


## 2. Load the Full-Information Modeling Data

The notebook first looks for saved Session 22 arrays. If they are not
available, it reconstructs the modeling data from a processed CSV file.


In [ ]:
array_candidates = [
    PROJECT_ROOT / "data" / "processed" / "session22_modeling_arrays.npz",
    PROJECT_ROOT / "data" / "processed" / "modeling_arrays.npz",
    PROJECT_ROOT / "data" / "processed" / "train_test_arrays.npz",
]

array_path = next(
    (path for path in array_candidates if path.exists()),
    None,
)

if array_path is not None:
    saved_arrays = np.load(array_path, allow_pickle=True)

    required_keys = {"Xtr_f", "Xte_f", "ytr", "yte"}
    missing_keys = required_keys.difference(saved_arrays.files)

    if missing_keys:
        raise KeyError(
            f"{array_path} is missing: {sorted(missing_keys)}"
        )

    Xtr_f = saved_arrays["Xtr_f"]
    Xte_f = saved_arrays["Xte_f"]
    ytr = saved_arrays["ytr"]
    yte = saved_arrays["yte"]

    print("Loaded Session 22 arrays from:")
    print(array_path)

else:
    csv_candidates = [
        PROJECT_ROOT / "data" / "processed" / "student_performance_clean.csv",
        PROJECT_ROOT / "data" / "processed" / "student_performance.csv",
        PROJECT_ROOT / "data" / "processed" / "student-mat-clean.csv",
        PROJECT_ROOT / "data" / "processed" / "student-mat.csv",
    ]

    csv_path = next(
        (path for path in csv_candidates if path.exists()),
        None,
    )

    if csv_path is None:
        processed_csvs = sorted(
            (PROJECT_ROOT / "data" / "processed").glob("*.csv")
        )

        if processed_csvs:
            csv_path = processed_csvs[0]

    if csv_path is None:
        raise FileNotFoundError(
            "No Session 22 array file or processed CSV was found. "
            "Run the earlier data-preparation sessions first."
        )

    data = pd.read_csv(csv_path)

    target_candidates = ["G3", "final_grade", "FinalGrade", "target"]
    target_column = next(
        (column for column in target_candidates if column in data.columns),
        None,
    )

    if target_column is None:
        raise KeyError(
            "The regression target was not found. Expected G3, "
            "final_grade, FinalGrade, or target."
        )

    X_full = data.drop(columns=[target_column])
    y_full = pd.to_numeric(data[target_column], errors="raise")

    X_full = pd.get_dummies(
        X_full,
        drop_first=False,
        dtype=float,
    )

    X_full = X_full.replace([np.inf, -np.inf], np.nan)

    numeric_medians = X_full.median(numeric_only=True)
    X_full = X_full.fillna(numeric_medians).fillna(0)

    Xtr_f, Xte_f, ytr, yte = train_test_split(
        X_full,
        y_full,
        test_size=0.20,
        random_state=42,
    )

    print("Reconstructed modeling arrays from:")
    print(csv_path)

print("Xtr_f shape:", np.asarray(Xtr_f).shape)
print("Xte_f shape:", np.asarray(Xte_f).shape)
print("ytr length:", len(ytr))
print("yte length:", len(yte))


## 3. Validate the Modeling Arrays


In [ ]:
Xtr_f = np.asarray(Xtr_f, dtype=float)
Xte_f = np.asarray(Xte_f, dtype=float)
ytr = np.asarray(ytr, dtype=float).reshape(-1)
yte = np.asarray(yte, dtype=float).reshape(-1)

assert Xtr_f.ndim == 2
assert Xte_f.ndim == 2
assert Xtr_f.shape[0] == len(ytr)
assert Xte_f.shape[0] == len(yte)
assert Xtr_f.shape[1] == Xte_f.shape[1]
assert np.isfinite(Xtr_f).all()
assert np.isfinite(Xte_f).all()
assert np.isfinite(ytr).all()
assert np.isfinite(yte).all()

print("Session 25 modeling-array validation passed.")


## 4. Define the Regression Evaluation Helper


In [ ]:
def eval_reg(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
    }


print("eval_reg() is ready.")


## 5. Scale the Features

Scaling is fitted on the training data only. The same fitted scaler is
then applied to the test data. This prevents test-set information from
leaking into model training.


In [ ]:
scaler = StandardScaler()

Xtr_scaled = scaler.fit_transform(Xtr_f)
Xte_scaled = scaler.transform(Xte_f)

print("Training and test features were scaled correctly.")


## 6. Define and Train the Four Baseline Models


In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(
        alpha=1.0,
        max_iter=10000,
    ),
    "Elastic Net": ElasticNet(
        alpha=1.0,
        l1_ratio=0.5,
        max_iter=10000,
    ),
}

results = []
fitted_models = {}
test_predictions = {}

for model_name, model in models.items():
    model.fit(Xtr_scaled, ytr)
    predictions = model.predict(Xte_scaled)
    metrics = eval_reg(yte, predictions)

    fitted_models[model_name] = model
    test_predictions[model_name] = predictions

    results.append(
        {
            "Session": 25,
            "Model": model_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "R2": metrics["R2"],
        }
    )

    print(
        f"{model_name}: "
        f"MAE={metrics['MAE']:.4f}, "
        f"RMSE={metrics['RMSE']:.4f}, "
        f"R2={metrics['R2']:.4f}"
    )

print("All four baseline models were trained successfully.")


## 7. Rank the Baseline Models by RMSE


In [ ]:
baseline_rows = (
    pd.DataFrame(results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

baseline_rows.insert(
    2,
    "RMSE Rank",
    range(1, len(baseline_rows) + 1),
)

display(baseline_rows.round(4))


## 8. Identify the Current Leading Baseline


In [ ]:
best_row = baseline_rows.iloc[0]

print("Current leading baseline")
print("------------------------")
print("Model:", best_row["Model"])
print(f"MAE:  {best_row['MAE']:.4f}")
print(f"RMSE: {best_row['RMSE']:.4f}")
print(f"R2:   {best_row['R2']:.4f}")


## 9. Save and Update the Model-Comparison Tables


In [ ]:
session25_output_path = (
    TABLES_DIRECTORY
    / "session25_baseline_regression_rows.csv"
)

comparison_path = (
    TABLES_DIRECTORY
    / "model_comparison_table.csv"
)

baseline_rows.to_csv(
    session25_output_path,
    index=False,
)

if comparison_path.exists():
    comparison_table = pd.read_csv(comparison_path)

    if "Session" in comparison_table.columns:
        session_numbers = pd.to_numeric(
            comparison_table["Session"],
            errors="coerce",
        )

        comparison_table = comparison_table.loc[
            session_numbers != 25
        ].copy()
    elif "Model" in comparison_table.columns:
        comparison_table = comparison_table.loc[
            ~comparison_table["Model"].isin(
                baseline_rows["Model"]
            )
        ].copy()

    comparison_table = pd.concat(
        [comparison_table, baseline_rows],
        ignore_index=True,
        sort=False,
    )
else:
    comparison_table = baseline_rows.copy()

comparison_table.to_csv(
    comparison_path,
    index=False,
)

print("Session 25 table saved to:")
print(session25_output_path)

print("Model-comparison table saved to:")
print(comparison_path)


## 10. Compare Coefficient Behavior


In [ ]:
coefficient_summary = []

for model_name, model in fitted_models.items():
    coefficients = np.asarray(model.coef_).reshape(-1)

    coefficient_summary.append(
        {
            "Model": model_name,
            "Number of Coefficients": len(coefficients),
            "Mean Absolute Coefficient": float(
                np.mean(np.abs(coefficients))
            ),
            "Maximum Absolute Coefficient": float(
                np.max(np.abs(coefficients))
            ),
            "Zero or Near-Zero Coefficients": int(
                np.isclose(
                    coefficients,
                    0.0,
                    atol=1e-8,
                ).sum()
            ),
        }
    )

coefficient_summary_df = pd.DataFrame(
    coefficient_summary
)

display(coefficient_summary_df.round(4))


## 11. Create the RMSE Comparison Figure


In [ ]:
plot_data = baseline_rows.sort_values("RMSE")

plt.figure(figsize=(9, 5))

bars = plt.bar(
    plot_data["Model"],
    plot_data["RMSE"],
    color=["#315C8C", "#4F81BD", "#78A5D1", "#A8C6E3"],
)

plt.title(
    "Session 25 Baseline Regression Model Comparison"
)
plt.xlabel("Regression Model")
plt.ylabel("Test RMSE")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, plot_data["RMSE"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()

figure_path = (
    FIGURES_DIRECTORY
    / "session25_baseline_rmse.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Figure saved to:")
print(figure_path)


## 12. Reflection

If Ridge and Lasso perform almost the same as ordinary Linear
Regression, regularization is providing little immediate test-error
improvement under the current settings. The unregularized estimates
may already be sufficiently stable, multicollinearity may not be
seriously damaging prediction, or the default penalty strengths may
not be optimal.

Similar RMSE values do not mean that the models learned identical
coefficients. Ridge may shrink coefficients, while Lasso may set some
coefficients to zero. Model selection should therefore consider
cross-validation, coefficient stability, interpretability, and test
performance rather than RMSE alone.


## 13. Completion Validation


In [ ]:
required_models = {
    "Linear Regression",
    "Ridge",
    "Lasso",
    "Elastic Net",
}

assert len(baseline_rows) == 4
assert set(baseline_rows["Model"]) == required_models
assert baseline_rows[
    ["MAE", "RMSE", "R2"]
].notna().all().all()
assert set(baseline_rows["RMSE Rank"]) == {1, 2, 3, 4}

print("SESSION 25 NOTEBOOK VALIDATION PASSED")
